<a href="https://colab.research.google.com/github/hmaroum-dotcom/Scraping_Offers_Data/blob/main/Scraping_Offres_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Scraper d’offres Data/BI (Luxembourg & Metz) pour Google Colab**

Le script contient deux cellules et deux sources :

LinkedIn Jobs (par défaut) : pages publiques, sans connexion. Les sélecteurs CSS utilisés sont connus et stables.
Moovijob (optionnel, désactivé par défaut) : je ne peux pas vérifier leur HTML actuel, donc les sélecteurs sont à valider (voir les notes).

**Cellule 1 : Installation des dépendances**

In [ ]:
# ============================================================
# CELLULE 1 : INSTALLATION (à exécuter une seule fois par session)
# ============================================================
# On retire les stubs snap défaillants
!apt-get remove -y -qq chromium-browser chromium-chromedriver > /dev/null 2>&1

# Vrai Google Chrome (.deb officiel)
!apt-get update -qq
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq /tmp/chrome.deb

# Dépendances Python (pandas reste sur la version imposée par Colab)
!pip install -q -U selenium beautifulsoup4 webdriver-manager
!pip install -q "pandas==2.2.3"

# Vérification : doit afficher "Google Chrome 1xx.x.x.x"
!google-chrome --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatk-bridge2.0-0t64:amd64.
Preparing to unpack .../3-libatk-bridge2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk-bridge2.0-0t64:amd64 (2.52.0-1b

**Cellule 2 : scraping, filtrage, scoring, export**

In [ ]:
# ============================================================
# CELLULE 2 : SCRAPING + FILTRAGE + SCORING + EXPORT CSV
# ============================================================
import re
import time
import random
import subprocess
import unicodedata
from urllib.parse import quote_plus

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

# ------------------------------------------------------------
# 1. PARAMÈTRES
# ------------------------------------------------------------
KEYWORDS = ["Data Engineer", "Consultant BI", "Business Intelligence",
            "Analytics Engineer", "Data Analyst Senior"]

# Zones de recherche envoyées à la plateforme
SEARCH_LOCATIONS = ["Luxembourg", "Metz, Grand Est, France"]

# Filtre géographique strict (comparé après normalisation sans accents/minuscules)
GEO_TERMS = ["luxembourg", "metz", "grand est", "teletravail", "remote"]

# Exclusions strictes (regex avec limites de mots)
EXCLUDE_PATTERNS = [r"\bstages?\b", r"\balternance\b", r"\bapprenti\w*",
                    r"\bjuniors?\b", r"\binternships?\b"]

# Scoring officiel : +3 / +2 / +1
SCORE_TECH = 3      # Power BI, SSIS, SAP BO, SQL
SCORE_SECTOR = 2    # Banque, Risques, Finance, Crédit
SCORE_ENGLISH = 1   # Anglais / English
PER_KEYWORD = False # False = +3 une seule fois si au moins un terme tech présent
                    # True  = +3 par terme distinct trouvé (plus discriminant)

TECH_TERMS = [r"power\s?bi", r"ssis", r"sap\s?bo", r"business\s?objects?", r"\bsql\b"]
SECTOR_TERMS = [r"banque", r"bancaire", r"\bbank\w*", r"risques?", r"\brisk\w*",
                r"financ\w+", r"credit\w*"]
ENGLISH_TERMS = [r"anglais", r"english"]

# --- Pertinence du titre : une offre doit contenir au moins un de ces termes ---
TITLE_INCLUDE = [r"\bdata\b", r"donnees", r"\bbi\b", r"business\s?intelligence",
                 r"analytics?", r"power\s?bi", r"\bsql\b", r"\betl\b", r"\bssis\b",
                 r"decisionnel", r"reporting", r"dwh", r"data\s?warehouse",
                 r"business\s?objects?", r"sap\s?bo", r"databricks", r"fabric",
                 r"\bdax\b", r"bigquery", r"azure"]

# Titres à écarter (rôles hors profil Data/BI engineering)
# Retirez un terme si le rôle vous intéresse (ex: data\s?governance)
TITLE_EXCLUDE = [r"scientist", r"\bgis\b", r"geospatial", r"director", r"\bpmo\b",
                 r"product\s?manager", r"data\s?governance", r"\bcompliance\b"]

# Bonus de départage : technos de votre profil
BONUS_TERMS = [r"fabric", r"databricks", r"data\s?factory", r"\badf\b", r"bigquery",
               r"oracle", r"\bdax\b", r"power\s?query", r"dwh", r"data\s?warehouse",
               r"modelisation", r"multidimensionnel", r"recouvrement"]

# Options de scraping
MAX_SCROLLS = 6             # nb de "scroll / voir plus" par recherche LinkedIn
FETCH_DESCRIPTION = True    # True = ouvre chaque offre pertinente pour lire sa description
MAX_DETAIL_PAGES = 150      # plafond de pages de détail ouvertes (≈ 10 min)
ENABLE_MOOVIJOB = False     # voir note sur les sélecteurs Moovijob
OUTPUT_CSV = "jobs_lux_bi.csv"


# ------------------------------------------------------------
# 2. OUTILS
# ------------------------------------------------------------
def normalize(text: str) -> str:
    """Minuscules + suppression des accents pour des comparaisons fiables."""
    text = unicodedata.normalize("NFKD", text or "")
    return "".join(c for c in text if not unicodedata.combining(c)).lower()


def pause(a=1.5, b=3.5):
    """Pause aléatoire pour imiter un comportement humain et limiter le blocage."""
    time.sleep(random.uniform(a, b))


def build_driver():
    """Chrome headless pour Colab : Google Chrome + chromedriver géré par Selenium Manager."""
    # Version réelle de Chrome, pour un user-agent cohérent
    out = subprocess.run(["google-chrome", "--version"], capture_output=True, text=True).stdout
    m = re.search(r"(\d+)\.", out)
    major = m.group(1) if m else "130"

    opts = Options()
    opts.binary_location = "/usr/bin/google-chrome"
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")                 # indispensable sur Colab (root)
    opts.add_argument("--disable-dev-shm-usage")      # /dev/shm trop petit sur Colab
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=fr-FR")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        f"(KHTML, like Gecko) Chrome/{major}.0.0.0 Safari/537.36"
    )

    try:
        # Selenium Manager télécharge le chromedriver correspondant à Chrome
        driver = webdriver.Chrome(options=opts)
    except Exception as e:
        print(f"Selenium Manager a échoué ({e.__class__.__name__}), repli sur webdriver-manager")
        from webdriver_manager.chrome import ChromeDriverManager
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

    # Masque navigator.webdriver
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    driver.set_page_load_timeout(40)
    return driver


# ------------------------------------------------------------
# 3. SCRAPING LINKEDIN (pages publiques "guest")
# ------------------------------------------------------------
def scrape_linkedin(driver, keyword: str, location: str) -> list[dict]:
    """Récupère les cartes d'offres LinkedIn pour un mot-clé et une zone."""
    url = (
        "https://www.linkedin.com/jobs/search?"
        f"keywords={quote_plus(keyword)}&location={quote_plus(location)}"
        "&f_TPR=r2592000"          # offres des 30 derniers jours
        "&f_E=3%2C4"               # niveaux Confirmé + Senior (retirer pour élargir)
        "&position=1&pageNum=0"
    )
    driver.get(url)
    pause(3, 5)

    # Détection du mur d'authentification (fréquent depuis les IP cloud)
    if "authwall" in driver.current_url or "/login" in driver.current_url:
        print(f"  ⚠️ Authwall LinkedIn pour '{keyword}' / '{location}' → ignoré")
        return []

    # Scroll + clic sur "Voir plus d'offres" pour charger davantage de résultats
    for _ in range(MAX_SCROLLS):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        pause(1.5, 2.5)
        try:
            btn = driver.find_element(By.CSS_SELECTOR, "button.infinite-scroller__show-more-button")
            if btn.is_displayed():
                btn.click()
                pause(1.5, 2.5)
        except Exception:
            pass

    soup = BeautifulSoup(driver.page_source, "html.parser")
    jobs = []
    for card in soup.select("div.base-search-card"):
        title = card.select_one("h3.base-search-card__title")
        company = card.select_one("h4.base-search-card__subtitle")
        loc = card.select_one("span.job-search-card__location")
        link = card.select_one("a.base-card__full-link")
        date = card.select_one("time")
        if not (title and link):
            continue
        jobs.append({
            "source": "LinkedIn",
            "titre": title.get_text(strip=True),
            "entreprise": company.get_text(strip=True) if company else "",
            "lieu": loc.get_text(strip=True) if loc else "",
            "date": date.get("datetime", "") if date else "",
            "description": "",
            "lien": link["href"].split("?")[0],
            "requete": f"{keyword} | {location}",
        })
    print(f"  ✔ LinkedIn '{keyword}' / '{location}' : {len(jobs)} cartes")
    return jobs


def scrape_with_retry(driver, kw, loc, tries=3):
    """Relance la recherche si LinkedIn renvoie 0 carte (limitation temporaire)."""
    for k in range(tries):
        jobs = scrape_linkedin(driver, kw, loc)
        if jobs:
            return jobs
        if k < tries - 1:
            print(f"  ↻ 0 carte, nouvelle tentative {k + 2}/{tries} dans ~30 s")
            pause(25, 40)
    return []


def fetch_description(driver, url: str) -> str:
    """Ouvre la page de détail d'une offre LinkedIn et renvoie le texte de description."""
    try:
        driver.get(url)
        pause(2, 4)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        block = soup.select_one("div.show-more-less-html__markup")
        return block.get_text(" ", strip=True) if block else ""
    except Exception:
        return ""


# ------------------------------------------------------------
# 4. SCRAPING MOOVIJOB (OPTIONNEL – SÉLECTEURS À VÉRIFIER)
# ------------------------------------------------------------
def scrape_moovijob(driver, keyword: str) -> list[dict]:
    """
    Version générique : à adapter après inspection de la page (F12).
    1) Ouvrez Moovijob dans votre navigateur, lancez une recherche, copiez l'URL
       de résultats et remplacez MOOVIJOB_SEARCH_URL ci-dessous.
    2) Ajustez les sélecteurs CSS dans la boucle selon le HTML réel.
    """
    MOOVIJOB_SEARCH_URL = "https://www.moovijob.com/fr/offres-emploi?q={kw}"  # À VÉRIFIER
    driver.get(MOOVIJOB_SEARCH_URL.format(kw=quote_plus(keyword)))
    pause(3, 5)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    jobs = []
    for card in soup.select("article, div.job-card, li.job"):          # À ADAPTER
        a = card.select_one("a[href]")
        title = card.select_one("h2, h3")
        if not (a and title):
            continue
        href = a["href"]
        if href.startswith("/"):
            href = "https://www.moovijob.com" + href
        jobs.append({
            "source": "Moovijob",
            "titre": title.get_text(strip=True),
            "entreprise": "",
            "lieu": "Luxembourg",   # Moovijob est centré Luxembourg ; affinez si le lieu est dispo
            "date": "",
            "description": card.get_text(" ", strip=True)[:500],
            "lien": href,
            "requete": keyword,
        })
    print(f"  ✔ Moovijob '{keyword}' : {len(jobs)} cartes")
    return jobs


# ------------------------------------------------------------
# 5. FILTRAGE ET SCORING
# ------------------------------------------------------------
def is_excluded(text: str) -> bool:
    """Vrai si l'offre contient un terme exclu (stage, alternance, junior...)."""
    t = normalize(text)
    return any(re.search(p, t) for p in EXCLUDE_PATTERNS)


def matches_geo(text: str) -> bool:
    """Vrai si lieu/titre contient Luxembourg, Metz, Grand Est ou Remote."""
    t = normalize(text)
    return any(term in t for term in GEO_TERMS)


def count_hits(patterns: list[str], text: str) -> int:
    """Nombre de motifs distincts trouvés dans le texte."""
    return sum(1 for p in patterns if re.search(p, text))


def compute_score(title: str, description: str) -> tuple[int, int]:
    """Score officiel (+3/+2/+1) sur titre + description, et bonus de départage."""
    t = normalize(f"{title} {description}")
    tech = count_hits(TECH_TERMS, t)
    sector = count_hits(SECTOR_TERMS, t)
    english = count_hits(ENGLISH_TERMS, t)
    if PER_KEYWORD:
        score = tech * SCORE_TECH + sector * SCORE_SECTOR + (SCORE_ENGLISH if english else 0)
    else:
        score = ((SCORE_TECH if tech else 0)
                 + (SCORE_SECTOR if sector else 0)
                 + (SCORE_ENGLISH if english else 0))
    return score, count_hits(BONUS_TERMS, t)


# ------------------------------------------------------------
# 6. EXÉCUTION PRINCIPALE
# ------------------------------------------------------------
driver = build_driver()
raw_jobs = []
candidates = []
try:
    # --- LinkedIn (avec relance si 0 carte) ---
    for kw in KEYWORDS:
        for loc in SEARCH_LOCATIONS:
            try:
                raw_jobs += scrape_with_retry(driver, kw, loc)
            except Exception as e:
                print(f"  ✖ Erreur LinkedIn '{kw}' / '{loc}' : {e}")
            pause(2, 4)

    # --- Moovijob (optionnel) ---
    if ENABLE_MOOVIJOB:
        for kw in KEYWORDS:
            try:
                raw_jobs += scrape_moovijob(driver, kw)
            except Exception as e:
                print(f"  ✖ Erreur Moovijob '{kw}' : {e}")
            pause(2, 4)

    # Dédoublonnage par lien
    raw_jobs = list({j["lien"]: j for j in raw_jobs}.values())
    print(f"\n{len(raw_jobs)} offres uniques collectées")

    # Pré-filtre AVANT d'ouvrir les pages : exclusion + titres hors profil + géo + pertinence
    for j in raw_jobs:
        if is_excluded(j["titre"]):
            continue
        if any(re.search(p, normalize(j["titre"])) for p in TITLE_EXCLUDE):
            continue
        if not matches_geo(f"{j['lieu']} {j['titre']}"):
            continue
        if not any(re.search(p, normalize(j["titre"])) for p in TITLE_INCLUDE):
            continue
        candidates.append(j)
    print(f"{len(candidates)} offres pertinentes après pré-filtre sur le titre")

    # Lecture des descriptions (offres les plus récentes d'abord)
    if FETCH_DESCRIPTION:
        candidates.sort(key=lambda j: j["date"], reverse=True)
        empties = 0
        for i, j in enumerate(candidates[:MAX_DETAIL_PAGES], 1):
            if j["source"] != "LinkedIn":
                continue
            j["description"] = fetch_description(driver, j["lien"])
            empties = empties + 1 if not j["description"] else 0
            if i % 20 == 0:
                print(f"  … {i} descriptions lues")
            if empties >= 8:      # 8 échecs d'affilée = probable blocage LinkedIn
                print("  ⚠️ Descriptions vides à répétition : arrêt (blocage probable)")
                break
finally:
    driver.quit()

# Offres écartées par le filtre stage/junior sur la description (pour vérification)
dropped = [f"{j['titre']} | {j['entreprise']}" for j in candidates
           if is_excluded(f"{j['titre']} {j['description']}")]
print(f"\n{len(dropped)} offres exclues (stage/junior/alternance...) :", *dropped, sep="\n  - ")

# Filtrage final et scoring
qualified = []
for j in candidates:
    if is_excluded(f"{j['titre']} {j['description']}"):
        continue
    j["score"], j["bonus"] = compute_score(j["titre"], j["description"])
    j["total"] = j["score"] + j["bonus"]
    qualified.append(j)

# DataFrame trié : total (score + bonus), puis score officiel, puis date la plus récente
cols = ["total", "score", "bonus", "titre", "entreprise", "lieu", "date", "source", "lien", "requete"]
df = pd.DataFrame(qualified, columns=cols + ["description"])
if not df.empty:
    df = df.sort_values(["total", "score", "date"],
                        ascending=[False, False, False]).reset_index(drop=True)

# Export CSV (utf-8-sig pour une ouverture correcte des accents dans Excel)
df[cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ {len(df)} offres qualifiées exportées dans '{OUTPUT_CSV}'")

# Affichage et téléchargement depuis Colab
pd.set_option("display.max_colwidth", 60)
display(df[cols].head(30))
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except Exception:
    pass

  ✔ LinkedIn 'Data Engineer' / 'Luxembourg' : 120 cartes
  ✔ LinkedIn 'Data Engineer' / 'Metz, Grand Est, France' : 33 cartes
  ✔ LinkedIn 'Consultant BI' / 'Luxembourg' : 120 cartes
  ✔ LinkedIn 'Consultant BI' / 'Metz, Grand Est, France' : 22 cartes
  ✔ LinkedIn 'Business Intelligence' / 'Luxembourg' : 120 cartes
  ✔ LinkedIn 'Business Intelligence' / 'Metz, Grand Est, France' : 30 cartes
  ✔ LinkedIn 'Analytics Engineer' / 'Luxembourg' : 120 cartes
  ✔ LinkedIn 'Analytics Engineer' / 'Metz, Grand Est, France' : 20 cartes
  ✔ LinkedIn 'Data Analyst Senior' / 'Luxembourg' : 120 cartes
  ✔ LinkedIn 'Data Analyst Senior' / 'Metz, Grand Est, France' : 20 cartes

314 offres uniques collectées
98 offres pertinentes après pré-filtre sur le titre
  … 20 descriptions lues
  … 40 descriptions lues
  … 60 descriptions lues
  … 80 descriptions lues

23 offres exclues (stage/junior/alternance...) :
  - Data Engineer (m/f/d) (réf. G26000171) | Police Luxembourg
  - Data Engineer (m/f/d) (réf. G260

,total,score,bonus,titre,entreprise,lieu,date,source,lien,requete
0,9,6,3,Digital Factory - Data Engineer - Supervising Associate,EY,"Luxembourg, Luxembourg, Luxembourg",2026-10-08,LinkedIn,https://lu.linkedin.com/jobs/view/digital-factory-data-e...,Analytics Engineer | Luxembourg
1,8,6,2,Senior Data Engineer,Devoteam,"Koerich, Canton of Capellen, Luxembourg",2026-10-14,LinkedIn,https://lu.linkedin.com/jobs/view/senior-data-engineer-a...,Analytics Engineer | Luxembourg
2,8,6,2,Senior Data Analyst,Netcompany,"Luxembourg, Luxembourg, Luxembourg",2026-10-06,LinkedIn,https://lu.linkedin.com/jobs/view/senior-data-analyst-at...,Data Analyst Senior | Luxembourg
3,8,6,2,Data Business Analyst (H/F),Davidson consulting,Luxembourg,2026-10-06,LinkedIn,https://lu.linkedin.com/jobs/view/data-business-analyst-...,Data Analyst Senior | Luxembourg
4,8,6,2,BI and Data Analytics Solutions Engineer (m/f),Experis IT Luxembourg,"Luxembourg, Luxembourg",2026-09-30,LinkedIn,https://lu.linkedin.com/jobs/view/bi-and-data-analytics-...,Data Analyst Senior | Luxembourg
5,8,4,4,Digital Factory - Senior Data Engineer - Senior Associate,EY,"Luxembourg, Luxembourg, Luxembourg",2026-10-17,LinkedIn,https://lu.linkedin.com/jobs/view/digital-factory-senior...,Analytics Engineer | Luxembourg
6,7,6,1,CONSULTANT SENIOR (H/F) DATA MANAGEMENT,TNP Luxembourg,"Luxembourg, Luxembourg",2026-10-20,LinkedIn,https://lu.linkedin.com/jobs/view/consultant-senior-h-f-...,Data Analyst Senior | Luxembourg
7,7,6,1,Data Engineer,EDAN Luxembourg,"Luxembourg, Luxembourg",2026-10-19,LinkedIn,https://lu.linkedin.com/jobs/view/data-engineer-at-edan-...,Analytics Engineer | Luxembourg
8,7,6,1,Data Engineer DBT/Snowflake,Talan,"Luxembourg, Luxembourg, Luxembourg",2026-10-15,LinkedIn,https://lu.linkedin.com/jobs/view/data-engineer-dbt-snow...,Analytics Engineer | Luxembourg
9,7,6,1,Analytics Solution Officer - based in Luxembourg,European Investment Bank (EIB),"Luxembourg, Luxembourg, Luxembourg",2026-10-03,LinkedIn,https://lu.linkedin.com/jobs/view/analytics-solution-off...,Business Intelligence | Luxembourg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>